<center><a href="https://www.nvidia.com/en-us/training/"><img src="images/DLI_Header.png" width="400" height="186" /></a></center>

## License

Unless otherwise noted:

- Code cells are licensed under the
  [Apache License 2.0](../../LICENSES/Apache-2.0.txt).
- Markdown and raw cells are licensed under the
  [Creative Commons Attribution 4.0 International License](../../LICENSES/CC-BY-4.0.txt).
- Third-party material retains its original license and is identified in
  [`THIRD_PARTY_NOTICES.md`](../../THIRD_PARTY_NOTICES.md).

Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES.

# 6. MNIST Classifier-Free Diffusion

This optional capstone trains a conditional diffusion model to generate handwritten digits from the MNIST dataset.

## Learning objectives

By the end of this notebook, you will be able to:

- reuse a diffusion pipeline with a new image dataset;
- train a class-conditional U-Net with random context dropout; and
- compare several classifier-free guidance weights while sampling digits.

## 6.1 Load MNIST

MNIST contains grayscale images of the digits zero through nine. Torchvision downloads the dataset automatically the first time this section runs. Unlike FashionMNIST, horizontal flipping is not appropriate because mirrored digits are generally not valid training examples.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision.utils import make_grid

from utils import UNet_utils, ddpm_utils, other_utils

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
IMG_SIZE = 28
IMG_CH = 1
BATCH_SIZE = 128
N_CLASSES = 10
INPUT_SIZE = (IMG_CH, IMG_SIZE, IMG_SIZE)

data_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

dataset = torchvision.datasets.MNIST(
    "./data/",
    download=True,
    train=True,
    transform=data_transform,
)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    pin_memory=device.type == "cuda",
)

print(f"Training examples: {len(dataset):,}")

In [ ]:
sample_images, sample_labels = next(iter(dataloader))
grid = make_grid(sample_images[:20], nrow=10, padding=2)
other_utils.show_tensor_image(grid)
plt.title("MNIST training examples")
plt.axis("off")
plt.show()

## 6.2 Configure the diffusion process

The forward process progressively combines an image with Gaussian noise according to a beta schedule. The shared `DDPM` helper calculates the cumulative schedule and implements both forward and reverse diffusion.

In [ ]:
T = 150
B_start = 0.0001
B_end = 0.02
B = torch.linspace(B_start, B_end, T, device=device)
ddpm = ddpm_utils.DDPM(B, device)

In [ ]:
x_0 = dataset[0][0].unsqueeze(0).to(device)
timesteps = [0, 20, 50, 90, 120, T - 1]

fig, axes = plt.subplots(1, len(timesteps), figsize=(12, 2.4))
for ax, timestep in zip(axes, timesteps):
    t = torch.tensor([timestep], device=device)
    x_t, _ = ddpm.q(x_0, t)
    ax.imshow(x_t.squeeze().cpu(), cmap="gray")
    ax.set_title(f"t = {timestep}")
    ax.axis("off")

plt.tight_layout()
plt.show()

The digit should become progressively less recognizable as the timestep increases. During training, the U-Net learns to predict the noise that produced each noisy image.

## 6.3 Create a conditional U-Net

The model uses the same reusable U-Net blocks introduced earlier in the course. Eager execution is used for portability across GPU, driver, and Triton versions.

In [ ]:
model = UNet_utils.UNet(
    T,
    IMG_CH,
    IMG_SIZE,
    down_chs=(64, 64, 128),
    t_embed_dim=8,
    c_embed_dim=N_CLASSES,
).to(device)

print("Number of parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
def get_context_mask(labels, drop_prob):
    return UNet_utils.get_context_mask(
        labels,
        drop_prob,
        num_classes=N_CLASSES,
        device=device,
    )


def get_loss(model, x_0, t, c_hot, c_mask):
    x_noisy, noise = ddpm.q(x_0, t)
    noise_pred = model(x_noisy, t, c_hot, c_mask)
    return F.mse_loss(noise_pred, noise)

## 6.4 Train the model

For a random fraction of examples, the context mask hides the digit label. This teaches the same network to make conditional and unconditional predictions, which are combined during classifier-free guidance. Training time depends on your hardware and can take several minutes.

In [ ]:
def train_mnist(model, dataloader, epochs=5, c_drop_prob=0.1, learning_rate=1e-3):
    optimizer = Adam(model.parameters(), lr=learning_rate)
    epoch_losses = []
    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in dataloader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            t = torch.randint(0, T, (images.size(0),), device=device).float()
            c_hot, c_mask = get_context_mask(labels, c_drop_prob)

            optimizer.zero_grad(set_to_none=True)
            loss = get_loss(model, images, t, c_hot, c_mask)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        mean_loss = running_loss / len(dataloader)
        epoch_losses.append(mean_loss)
        print(f"Epoch {epoch + 1:02d}/{epochs:02d} | loss: {mean_loss:.5f}")

    return epoch_losses

In [ ]:
losses = train_mnist(model, dataloader)

In [ ]:
plt.plot(range(1, len(losses) + 1), losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Mean training loss")
plt.title("MNIST diffusion training")
plt.show()

## 6.5 Sample with classifier-free guidance

For each digit, the sampler predicts noise with and without its class context. A guidance weight $w$ combines the predictions:

$$\epsilon_w = (1 + w)\epsilon_{conditioned} - w\epsilon_{unconditioned}$$

The following cell generates one row for each guidance weight and one column for each digit.

In [ ]:
@torch.no_grad()
def generate_digit_grid(model, guidance_weights=(0.0, 1.0, 2.0)):
    model.eval()
    digits = torch.arange(N_CLASSES, device=device)
    digit_context = F.one_hot(digits, num_classes=N_CLASSES).float()
    samples, _ = ddpm_utils.sample_w(
        model,
        ddpm,
        INPUT_SIZE,
        T,
        digit_context,
        device,
        w_tests=list(guidance_weights),
        store_freq=T,
    )
    return samples


guidance_weights = (0.0, 1.0, 2.0)
generated_digits = generate_digit_grid(model, guidance_weights)
grid = make_grid(generated_digits.cpu(), nrow=N_CLASSES, padding=2, normalize=True)

plt.figure(figsize=(14, 4.5))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.title("Rows: guidance weights 0.0, 1.0, and 2.0; columns: digits 0 through 9")
plt.axis("off")
plt.show()

## 6.6 Continue experimenting

Try changing the guidance weights, number of epochs, beta schedule, or context-drop probability. Compare image clarity and variety rather than optimizing for a grader. You can also replace MNIST with another small labeled image dataset and reuse the same training loop.

<center><a href="https://www.nvidia.com/en-us/training/"><img src="images/DLI_Header.png" width="400" height="186" /></a></center>